In [3]:
import pandas as pd 
import openpyxl

In [4]:
df = pd.read_excel("ventas-clasificacion_limpio.xlsx") 


In [5]:
#---#
# CELDA 1: Imports y carga del dataset
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

df = pd.read_excel('ventas-clasificacion_limpio.xlsx')

print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head()

#---#
# CELDA 2: Info general - tipos de datos y nulos
df.info()
print("\nValores nulos por columna:")
df.isna().sum().sort_values(ascending=False)

#---#
# CELDA 3: Limpieza y normalización de columnas clave
# - Normalizamos "Estado Pip." (hay duplicados por mayúscula/género: "En Ejecución" / "En ejecución", "Cancelado"/"Cancelada", etc.)
# - Convertimos "Duración" (formato "H:MM") a minutos numéricos
# - Nos aseguramos que las fechas sean datetime

df['Estado_norm'] = df['Estado Pip.'].str.strip().str.lower()
mapa_estados = {
    'en ejecución': 'en ejecucion',
    'cancelada': 'cancelado',
    'finalizada': 'finalizado',
}
df['Estado_norm'] = df['Estado_norm'].replace(mapa_estados)

def duracion_a_minutos(valor):
    try:
        h, m = str(valor).split(':')
        return int(h) * 60 + int(m)
    except Exception:
        return np.nan

df['Duracion_min'] = df['Duración'].apply(duracion_a_minutos)

df['Fecha de Inicio'] = pd.to_datetime(df['Fecha de Inicio'], errors='coerce')
df['Fecha de Cierre'] = pd.to_datetime(df['Fecha de Cierre'], errors='coerce')

df[['Estado Pip.', 'Estado_norm', 'Duración', 'Duracion_min']].head()

#---#
# CELDA 4: Análisis por CUENTA - volumen de tickets
tickets_por_cuenta = df['Cuenta'].value_counts()

print(f"Cuentas distintas: {df['Cuenta'].nunique()}")
print("\nTop 15 cuentas con más tickets:")
tickets_por_cuenta.head(15)

#---#
# CELDA 5: Análisis por CUENTA - duración promedio y total dedicada
resumen_cuenta = df.groupby('Cuenta').agg(
    tickets=('Nº', 'count'),
    duracion_promedio_min=('Duracion_min', 'mean'),
    duracion_total_min=('Duracion_min', 'sum')
).sort_values('tickets', ascending=False)

resumen_cuenta.head(15)

#---#
# CELDA 6: Análisis por TÉCNICO (columna "Asignado a")
tickets_por_tecnico = df['Asignado a'].value_counts()

print("Tickets atendidos por técnico:")
print(tickets_por_tecnico)

resumen_tecnico = df.groupby('Asignado a').agg(
    tickets=('Nº', 'count'),
    duracion_promedio_min=('Duracion_min', 'mean'),
    duracion_total_min=('Duracion_min', 'sum'),
    cuentas_distintas=('Cuenta', 'nunique')
).sort_values('tickets', ascending=False)

resumen_tecnico

#---#
# CELDA 7: Análisis por SECTOR / DEPARTAMENTO ("Departamento / Equipo") - tiempos promedio
resumen_sector = df.groupby('Departamento / Equipo').agg(
    tickets=('Nº', 'count'),
    duracion_promedio_min=('Duracion_min', 'mean'),
    duracion_mediana_min=('Duracion_min', 'median'),
    duracion_total_min=('Duracion_min', 'sum')
).sort_values('tickets', ascending=False)

resumen_sector

#---#
# CELDA 8: Distribución de ESTADOS de los tickets
estados = df['Estado_norm'].value_counts()
print(estados)
print(f"\n% Finalizados: {(estados.get('finalizado', 0) / len(df) * 100):.1f}%")
print(f"% Cancelados: {(estados.get('cancelado', 0) / len(df) * 100):.1f}%")
print(f"% Pendientes: {(estados.get('pendiente', 0) / len(df) * 100):.1f}%")

#---#
# CELDA 9: Análisis temporal - tickets por fecha
tickets_por_dia = df.groupby(df['Fecha de Inicio'].dt.date)['Nº'].count()
tickets_por_mes = df.groupby(df['Fecha de Inicio'].dt.to_period('M'))['Nº'].count()

print("Tickets por mes:")
print(tickets_por_mes)

print(f"\nRango de fechas: {df['Fecha de Inicio'].min().date()} a {df['Fecha de Inicio'].max().date()}")
print(f"Promedio de tickets por día activo: {tickets_por_dia.mean():.1f}")

#---#
# CELDA 10: Análisis temporal - día de la semana y hora de inicio con más carga
df['Dia_semana'] = df['Fecha de Inicio'].dt.day_name()
df['Hora_num'] = pd.to_datetime(df['Horas Inicio'], format='%H:%M', errors='coerce').dt.hour

orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
tickets_por_dia_semana = df['Dia_semana'].value_counts().reindex(orden_dias)
tickets_por_hora = df['Hora_num'].value_counts().sort_index()

print("Tickets por día de la semana:")
print(tickets_por_dia_semana)
print("\nTickets por hora del día:")
print(tickets_por_hora)

#---#
# CELDA 11: Tipo de origen y su relación con la duración
resumen_origen = df.groupby('Tipo de origen').agg(
    tickets=('Nº', 'count'),
    duracion_promedio_min=('Duracion_min', 'mean')
).sort_values('tickets', ascending=False)

resumen_origen

#---#
# CELDA 12: Tickets "improductivos" (duración 0 min) por técnico y por sector
# Útil para detectar tareas administrativas, cancelaciones rápidas, o registros sin trabajo real.
sin_duracion = df[df['Duracion_min'] == 0]

print(f"Tickets con 0 minutos de duración: {len(sin_duracion)} ({len(sin_duracion)/len(df)*100:.1f}% del total)")
print("\nPor técnico:")
print(sin_duracion['Asignado a'].value_counts())
print("\nPor estado:")
print(sin_duracion['Estado Pip.'].value_counts())

#---#
# CELDA 13: Cruce Cuenta x Técnico - quién atiende a cada cuenta (top 10 cuentas)
top_cuentas = tickets_por_cuenta.head(10).index
tabla_cruzada = pd.crosstab(df[df['Cuenta'].isin(top_cuentas)]['Cuenta'],
                             df[df['Cuenta'].isin(top_cuentas)]['Asignado a'])

tabla_cruzada

#---#
# CELDA 14: Resumen ejecutivo final
print("="*60)
print("RESUMEN EJECUTIVO")
print("="*60)
print(f"Total de tickets: {len(df)}")
print(f"Cuentas distintas atendidas: {df['Cuenta'].nunique()}")
print(f"Técnicos activos: {df['Asignado a'].nunique()}")
print(f"Duración promedio general: {df['Duracion_min'].mean():.1f} min")
print(f"Duración total acumulada: {df['Duracion_min'].sum() / 60:.1f} horas")
print(f"Período: {df['Fecha de Inicio'].min().date()} - {df['Fecha de Inicio'].max().date()}")
print(f"\nCuenta con más tickets: {tickets_por_cuenta.index[0]} ({tickets_por_cuenta.iloc[0]} tickets)")
print(f"Técnico con más tickets: {tickets_por_tecnico.index[0]} ({tickets_por_tecnico.iloc[0]} tickets)")
print(f"Sector con más tickets: {resumen_sector.index[0]} ({resumen_sector['tickets'].iloc[0]} tickets)")

Filas: 1758 | Columnas: 16
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1758 entries, 0 to 1757
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Fecha de Inicio        1758 non-null   datetime64[ns]
 1   Horas Inicio           1758 non-null   object        
 2   Cuenta                 1758 non-null   object        
 3   Título                 1758 non-null   object        
 4   Estado Pip.            1758 non-null   object        
 5   Fecha de Cierre        1399 non-null   datetime64[ns]
 6   Hora de cierre         1399 non-null   object        
 7   Nº                     1758 non-null   int64         
 8   Tipo de origen         1747 non-null   object        
 9   Origen                 1747 non-null   object        
 10  Id del origen          1747 non-null   float64       
 11  Departamento / Equipo  1735 non-null   object        
 12  Asignado a             1758 non-nul

In [7]:

"""
================================================================================
ANÁLISIS DE CARGA DE HORAS (horas.xlsx) + CRUCE CON CLASIFICACIÓN DE VENTAS
================================================================================
 
Este script hace tres cosas, cada una en su propia sección de funciones:
 
1. PURIFICAR   -> limpia el dataset 'horas.xlsx'
2. ANALIZAR    -> genera estadísticas generales del dataset ya limpio
3. CRUZAR      -> lo combina con 'ventas-clasificacion_limpio.xlsx' usando
                  el número de caso como clave común
 
Solo se usa pandas (y 're', que es librería estándar de Python, no un
paquete externo).
 
Cómo usarlo:
    python analisis_horas.py
 
Requiere que los dos archivos .xlsx estén en la misma carpeta que el script
(o ajustar las rutas en la sección "CONFIGURACIÓN" al final del archivo).
================================================================================
"""
 
import re
import pandas as pd
 
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
 
 
# ==============================================================================
# 1. PURIFICAR
# ==============================================================================
 
def cargar_horas(ruta_archivo):
    """Carga el archivo horas.xlsx tal cual viene."""
    df = pd.read_excel(ruta_archivo)
    return df
 
 
def eliminar_columnas_con_muchos_nulos(df, umbral_nulos=0.90, verbose=True):
    """
    Elimina columnas donde el % de valores nulos supera el umbral.
    umbral_nulos=0.90 significa: si una columna tiene más del 90% de nulos, se elimina.
    """
    porcentaje_nulos = df.isna().mean()
    columnas_a_eliminar = porcentaje_nulos[porcentaje_nulos > umbral_nulos].index.tolist()
 
    if verbose and columnas_a_eliminar:
        print(f"[Purificar] Columnas eliminadas por exceso de nulos (> {umbral_nulos:.0%}):")
        for col in columnas_a_eliminar:
            print(f"   - {col}: {porcentaje_nulos[col]:.1%} nulos")
 
    return df.drop(columns=columnas_a_eliminar)
 
 
def eliminar_columnas_baja_varianza(df, umbral_dominancia=0.98, verbose=True):
    """
    Elimina columnas "irrelevantes" porque casi no varían:
    - columnas con un solo valor único (constantes), o
    - columnas donde el valor más frecuente representa más del umbral_dominancia
      de las filas (ej: 98% de las filas tienen el mismo valor -> no aporta información).
    """
    columnas_a_eliminar = []
 
    for col in df.columns:
        valores_no_nulos = df[col].dropna()
        if valores_no_nulos.empty:
            continue  # ya la habrá eliminado el filtro de nulos, se ignora acá
 
        nunique = valores_no_nulos.nunique()
        if nunique <= 1:
            columnas_a_eliminar.append((col, "valor constante"))
            continue
 
        proporcion_valor_top = valores_no_nulos.value_counts(normalize=True).iloc[0]
        if proporcion_valor_top > umbral_dominancia:
            columnas_a_eliminar.append((col, f"{proporcion_valor_top:.1%} un solo valor"))
 
    if verbose and columnas_a_eliminar:
        print(f"[Purificar] Columnas eliminadas por baja varianza (> {umbral_dominancia:.0%} un mismo valor):")
        for col, motivo in columnas_a_eliminar:
            print(f"   - {col}: {motivo}")
 
    return df.drop(columns=[col for col, _ in columnas_a_eliminar])
 
 
def extraer_num_caso(df, columna_origen='Origen', nueva_columna='Num_caso'):
    """
    La columna 'Origen' trae texto tipo:
        'Caso: Configurar impresora - Nº: 16839'
        'Contrato: Mantenimiento web energyperf.com - Nº: 35'
 
    Se extrae el número que sigue a 'Nº:' y se guarda como entero en una
    columna nueva ('Num_caso'), que después sirve como clave para cruzar
    con el dataset de ventas/clasificación (columna 'Nº').
    """
    patron = re.compile(r'N[ºo°]?\s*:?\s*(\d+)', re.IGNORECASE)
 
    def _extraer(texto):
        match = patron.search(str(texto))
        return int(match.group(1)) if match else pd.NA
 
    df[nueva_columna] = df[columna_origen].apply(_extraer)
    return df
 
 
def duracion_a_minutos(valor):
    """Convierte una duración en formato 'H:MM' a minutos totales (int)."""
    try:
        h, m = str(valor).split(':')
        return int(h) * 60 + int(m)
    except (ValueError, AttributeError):
        return pd.NA
 
 
def purificar_horas(ruta_archivo, umbral_nulos=0.90, umbral_dominancia=0.98):
    """
    Orquesta toda la limpieza de horas.xlsx:
      1) carga el archivo
      2) elimina columnas con demasiados nulos
      3) elimina columnas de baja varianza / irrelevantes
      4) extrae Num_caso desde 'Origen'
      5) convierte 'Duración' a minutos numéricos
    Devuelve el DataFrame limpio.
    """
    print("=" * 70)
    print("PURIFICANDO horas.xlsx")
    print("=" * 70)
 
    df = cargar_horas(ruta_archivo)
    filas_original, columnas_original = df.shape
 
    df = eliminar_columnas_con_muchos_nulos(df, umbral_nulos)
    df = eliminar_columnas_baja_varianza(df, umbral_dominancia)
    df = extraer_num_caso(df)
 
    df['Duracion_min'] = df['Duración'].apply(duracion_a_minutos)
    df['Fecha de Inicio'] = pd.to_datetime(df['Fecha de Inicio'], errors='coerce', dayfirst=True)
    df['Fecha de Finalización'] = pd.to_datetime(df['Fecha de Finalización'], errors='coerce', dayfirst=True)
 
    print(f"\n[Purificar] Filas: {filas_original} -> {df.shape[0]}")
    print(f"[Purificar] Columnas: {columnas_original} -> {df.shape[1]}")
 
    sin_num_caso = df['Num_caso'].isna().sum()
    if sin_num_caso:
        print(f"[Purificar] Aviso: {sin_num_caso} filas sin Num_caso detectado en 'Origen'.")
 
    return df
 
 
# ==============================================================================
# 2. ANALIZAR
# ==============================================================================
 
def info_basica(df, nombre_dataset="dataset"):
    """Información general del dataset: tamaño, columnas, nulos restantes."""
    print("\n" + "-" * 70)
    print(f"INFO BÁSICA - {nombre_dataset}")
    print("-" * 70)
    print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
    print("\nColumnas y tipos de dato:")
    print(df.dtypes)
    nulos = df.isna().sum()
    nulos = nulos[nulos > 0].sort_values(ascending=False)
    if not nulos.empty:
        print("\nColumnas con nulos restantes:")
        print(nulos)
    else:
        print("\nSin valores nulos restantes.")
 
 
def resumen_por_tecnico(df, columna_tecnico='Usuario', columna_duracion='Duracion_min'):
    """Registros y horas por técnico ('Usuario' en horas.xlsx), ordenado de mayor a menor carga."""
    resumen = df.groupby(columna_tecnico).agg(
        registros=(columna_duracion, 'count'),
        minutos_totales=(columna_duracion, 'sum'),
        minutos_promedio=(columna_duracion, 'mean'),
    ).sort_values('registros', ascending=False)
 
    resumen['horas_totales'] = (resumen['minutos_totales'] / 60).round(1)
    resumen['minutos_promedio'] = resumen['minutos_promedio'].round(1)
 
    return resumen
 
 
def resumen_por_cuenta(df, columna_cuenta='Cuenta', columna_duracion='Duracion_min'):
    """Registros y horas por cuenta, ordenado de mayor a menor carga."""
    resumen = df.groupby(columna_cuenta).agg(
        registros=(columna_duracion, 'count'),
        minutos_totales=(columna_duracion, 'sum'),
        minutos_promedio=(columna_duracion, 'mean'),
    ).sort_values('registros', ascending=False)
 
    resumen['horas_totales'] = (resumen['minutos_totales'] / 60).round(1)
    resumen['minutos_promedio'] = resumen['minutos_promedio'].round(1)
 
    return resumen
 
 
def analizar_horas(df):
    """
    Corre el análisis completo sobre el dataset de horas ya purificado:
    info básica, resumen por técnico y resumen por cuenta. Imprime un
    resumen ejecutivo al final. Devuelve (resumen_tecnico, resumen_cuenta)
    por si se quieren reutilizar/exportar.
    """
    print("\n" + "=" * 70)
    print("ANALIZANDO horas.xlsx")
    print("=" * 70)
 
    info_basica(df, "horas.xlsx (purificado)")
 
    resumen_tecnico = resumen_por_tecnico(df)
    resumen_cuenta = resumen_por_cuenta(df)
 
    print("\nTop 10 técnicos por cantidad de registros:")
    print(resumen_tecnico.head(10)[['registros', 'horas_totales']])
 
    print("\nTop 10 técnicos por horas totales dedicadas:")
    print(resumen_tecnico.sort_values('horas_totales', ascending=False).head(10)[['registros', 'horas_totales']])
 
    print("\nTop 10 cuentas por cantidad de registros:")
    print(resumen_cuenta.head(10)[['registros', 'horas_totales']])
 
    print("\nTop 10 cuentas por horas totales dedicadas:")
    print(resumen_cuenta.sort_values('horas_totales', ascending=False).head(10)[['registros', 'horas_totales']])
 
    horas_totales = df['Duracion_min'].sum() / 60
    fecha_min = df['Fecha de Inicio'].min()
    fecha_max = df['Fecha de Inicio'].max()
 
    print("\n" + "-" * 70)
    print("RESUMEN EJECUTIVO - horas.xlsx")
    print("-" * 70)
    print(f"Total de registros: {len(df)}")
    print(f"Técnicos activos: {df['Usuario'].nunique()}")
    print(f"Cuentas distintas: {df['Cuenta'].nunique()}")
    print(f"Horas totales cargadas: {horas_totales:.1f} hs")
    print(f"Período: {fecha_min.date() if pd.notna(fecha_min) else 'N/D'} - "
          f"{fecha_max.date() if pd.notna(fecha_max) else 'N/D'}")
    print(f"Técnico con más registros: {resumen_tecnico.index[0]} ({resumen_tecnico['registros'].iloc[0]})")
    print(f"Técnico con más horas: "
          f"{resumen_tecnico.sort_values('horas_totales', ascending=False).index[0]} "
          f"({resumen_tecnico['horas_totales'].max():.1f} hs)")
    print(f"Cuenta con más registros: {resumen_cuenta.index[0]} ({resumen_cuenta['registros'].iloc[0]})")
    print(f"Cuenta con más horas: "
          f"{resumen_cuenta.sort_values('horas_totales', ascending=False).index[0]} "
          f"({resumen_cuenta['horas_totales'].max():.1f} hs)")
 
    return resumen_tecnico, resumen_cuenta
 
 
# ==============================================================================
# 3. CRUZAR
# ==============================================================================
 
def preparar_ventas(ruta_archivo):
    """
    Prepara ventas-clasificacion_limpio.xlsx siguiendo el mismo criterio del
    notebook original (normalización de 'Estado Pip.' y 'Duración' en minutos),
    para que sea compatible con el cruce contra horas.xlsx.
    """
    df = pd.read_excel(ruta_archivo)
 
    df['Estado_norm'] = df['Estado Pip.'].str.strip().str.lower()
    mapa_estados = {
        'en ejecución': 'en ejecucion',
        'cancelada': 'cancelado',
        'finalizada': 'finalizado',
    }
    df['Estado_norm'] = df['Estado_norm'].replace(mapa_estados)
 
    df['Duracion_min'] = df['Duración'].apply(duracion_a_minutos)
 
    df['Fecha de Inicio'] = pd.to_datetime(df['Fecha de Inicio'], errors='coerce')
    df['Fecha de Cierre'] = pd.to_datetime(df['Fecha de Cierre'], errors='coerce')
 
    return df
 
 
def cruzar_datasets(df_horas, df_ventas):
    """
    Cruza horas.xlsx (columna 'Num_caso') con ventas-clasificacion_limpio.xlsx
    (columna 'Nº'), que identifica el mismo caso en ambos datasets.
 
    Devuelve el DataFrame combinado (left join desde horas hacia ventas) y
    muestra un resumen de qué tan bien "pegó" el cruce.
    """
    print("\n" + "=" * 70)
    print("CRUZANDO horas.xlsx con ventas-clasificacion_limpio.xlsx")
    print("=" * 70)
 
    df_cruzado = df_horas.merge(
        df_ventas,
        how='left',
        left_on='Num_caso',
        right_on='Nº',
        suffixes=('_horas', '_ventas'),
    )
 
    # Ambos datasets traen una columna 'Nº' (numeración interna propia de cada
    # uno), por eso el merge las renombra con sufijos. La que indica si hubo
    # match es la que vino de ventas: 'Nº_ventas'.
    columna_match = 'Nº_ventas' if 'Nº_ventas' in df_cruzado.columns else 'Nº'
 
    total = len(df_cruzado)
    con_match = df_cruzado[columna_match].notna().sum()
 
    print(f"Registros de horas.xlsx: {total}")
    print(f"Con caso encontrado en ventas: {con_match} ({con_match / total * 100:.1f}%)")
    print(f"Sin match: {total - con_match} ({(total - con_match) / total * 100:.1f}%)")
 
    # Ejemplo de análisis cruzado: horas dedicadas por Departamento / Equipo
    # (columna que solo existe en ventas-clasificacion_limpio.xlsx)
    if 'Departamento / Equipo' in df_cruzado.columns:
        resumen_depto = df_cruzado.dropna(subset=['Departamento / Equipo']).groupby(
            'Departamento / Equipo'
        ).agg(
            registros=('Num_caso', 'count'),
            horas_totales=('Duracion_min_horas', lambda x: round(x.sum() / 60, 1)),
        ).sort_values('registros', ascending=False)
 
        print("\nHoras cargadas (horas.xlsx) por Departamento / Equipo (dato tomado de ventas):")
        print(resumen_depto)
 
    return df_cruzado
 
 
# ==============================================================================
# CONFIGURACIÓN / EJECUCIÓN
# ==============================================================================
 
RUTA_HORAS = 'horas.xlsx'
RUTA_VENTAS = 'ventas-clasificacion_limpio.xlsx'
 
 
def main():
    # 1) Purificar
    df_horas = purificar_horas(RUTA_HORAS)
 
    # 2) Analizar
    resumen_tecnico, resumen_cuenta = analizar_horas(df_horas)
 
    # 3) Cruzar
    df_ventas = preparar_ventas(RUTA_VENTAS)
    df_cruzado = cruzar_datasets(df_horas, df_ventas)
 
    return df_horas, df_ventas, df_cruzado, resumen_tecnico, resumen_cuenta
 

In [8]:
main()

PURIFICANDO horas.xlsx
[Purificar] Columnas eliminadas por exceso de nulos (> 90%):
   - Cliente ERP: 100.0% nulos
   - Tiempo de Viaje: 100.0% nulos
   - Horas no imputable: 100.0% nulos
   - Tipo de tiempo no imputable: 100.0% nulos
   - Razón de tiempo no imputable: 100.0% nulos
   - País: 100.0% nulos
   - Dirección: 100.0% nulos
   - Zona / Provincia / Municipio: 100.0% nulos
   - Ciudad: 100.0% nulos
   - Código Postal: 100.0% nulos
   - Latitud: 100.0% nulos
   - Longitud: 100.0% nulos
[Purificar] Columnas eliminadas por baja varianza (> 98% un mismo valor):
   - Estado de Cuenta: 99.6% un solo valor

[Purificar] Filas: 21167 -> 21167
[Purificar] Columnas: 31 -> 20

ANALIZANDO horas.xlsx

----------------------------------------------------------------------
INFO BÁSICA - horas.xlsx (purificado)
----------------------------------------------------------------------
Filas: 21167 | Columnas: 20

Columnas y tipos de dato:
Nº                                              int64
Fecha 

(          Nº     Fecha de Inicio Fecha de Finalización Tipo de fuente                                             Origen                         Cuenta  \
 0      21194 2026-07-13 14:49:00   2026-07-13 15:12:00           Caso             Caso: Configurar impresora - Nº: 16839                 Hierros Trotta   
 1      21193 2026-07-13 13:58:00   2026-07-13 14:15:00           Caso  Caso: Inconvenientes para descargar archivo - ...  L.F.R. Constructora Vial S.A.   
 2      21192 2026-07-13 14:00:00   2026-07-13 14:08:00           Caso       Caso: Inconvenientes con el mail - Nº: 16927  L.F.R. Constructora Vial S.A.   
 3      21191 2026-07-13 13:50:00   2026-07-13 14:00:00           Caso  Caso: Dar acceso a Braian Oliver a carpeta - N...  L.F.R. Constructora Vial S.A.   
 4      21189 2026-07-13 12:34:00   2026-07-13 12:40:00           Caso  Caso: Inconvenientes con archivos Excel - Nº: ...                    Hidroar S.A   
 ...      ...                 ...                   ...         